# Match labelled rows with LLM-extracted rows
1. Load data
2. Vectorize rows
3. Match rows 
4. Compute accuracy


In [49]:
import datetime as dt
import geopandas as gpd
from matplotlib import pyplot as plt
from src.data import *
from src.post_process_functions import *
from src.data_format import format_output, delistify_cols
from src.hazard_def import *
from src.impact_def import *
from src.accuracy import *
from src.geocoding_utils import clean_geometry
from src.utils import filter_by_flags, normalize_flags, get_flag_cols, print_impact

## Load data

In [50]:
# load data (model)
split_lowest = False
suffix = "_geo_split_lowest" if split_lowest else "_geo"
models = ["meta-llama_llama-4-scout-17b-16e-instruct", "llama-3.1-8b-instant", "llama-3.3-70b-versatile", "openai_gpt-oss-20b"]
# "llama-3.1-8b-instant"
# #"llama-3.3-70b-versatile"
# "meta-llama_llama-4-scout-17b-16e-instruct"
res_savename_ext = f"post_processed_0-39_labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v070726_v080726{suffix}"

# extracted_df_no_geo = pd.read_csv(DATA_OUT_PROC / (res_savename_ext+".csv"))

# load data (labelled)
res_savename_lab = f"post_processed_labelled_reports_all_v30062026_v300626{suffix}"  # merged_subtypes_

# labelled_df_no_geo = pd.read_csv(DATA_OUT_PROC / (res_savename_lab+".csv"))
# load geocoded data
extracted_df = gpd.read_file(DATA_OUT_PROC / (res_savename_ext+".gpkg"))#(res_savename_ext+suffix+suffix2+".gpkg")
# load data (labelled)
labelled_df = gpd.read_file(DATA_OUT_PROC / (res_savename_lab+".gpkg"))#(res_savename_lab+suffix+suffix2+".gpkg")

In [51]:
flag_cols = get_flag_cols(extracted_df)
flag_cols_lab = get_flag_cols(labelled_df)
extracted_df[flag_cols] = extracted_df[flag_cols].map(normalize_flags)
labelled_df[flag_cols_lab] = labelled_df[flag_cols_lab].map(normalize_flags)

In [52]:
flag_cols

['valid_errors_impactValue',
 'valid_errors_loc',
 'valid_errors_haz',
 'valid_errors_dates',
 'flag_value_not_in_text',
 'flag_missing_startYear',
 'flag_missing_endYear',
 'flag_missing_startMonth',
 'flag_missing_endMonth',
 'flag_missing_startDay',
 'flag_missing_endDay',
 'flag_startYear_after_endYear',
 'flag_inconsistent_year',
 'flag_inconsistent_month',
 'flag_inconsistent_day',
 'flag_impactSubtype_reclass',
 'flag_unknown_subtype',
 'flag_hazards_reclass',
 'flag_hazards_unknown',
 'flag_remove_number_unit',
 'flag_remove_number_unit_error',
 'flag_SI_unit_standardization',
 'flag_SI_unit_standardization_error',
 'flag_unit_harmonization',
 'flag_non_currency_unit_conversion',
 'flag_non_currency_unit_conversion_error',
 'flag_non-SI_unit_standardization_error',
 'flag_non-SI_unit_standardization',
 'flag_people_unit_normalization',
 'flag_currency_conversion',
 'flag_currency_conversion_error',
 'flag_reclass_subtype_from_unit',
 'flag_impactSubtype_merged',
 'flag_inferred

In [53]:
#reformat output
labelled_df = format_output(labelled_df)
extracted_df = format_output(extracted_df)


##for matching, need to replace NaNs in dates and units as np.nan cannot be compared
#abelled_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]] = labelled_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]].fillna(-1)
#xtracted_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]] = extracted_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]].fillna(-1)
#abelled_df["impactUnit"] = labelled_df["impactUnit"].fillna("null")
#xtracted_df["impactUnit"] = extracted_df["impactUnit"].fillna("null")

DEF_CRS_EPSG = "EPSG:4326"
labelled_df = labelled_df.set_crs(DEF_CRS_EPSG, allow_override=True)
extracted_df = extracted_df.set_crs(DEF_CRS_EPSG, allow_override=True)

#need to clean geometry column
labelled_df["geometry"] = labelled_df["geometry"].apply(clean_geometry)
extracted_df["geometry"] = extracted_df["geometry"].apply(clean_geometry)


In [54]:
extracted_df.shape

(980, 91)

In [55]:
labelled_df.shape

(770, 82)

In [56]:
filter_flags = [  # flags to filter out (i.e. keep only rows for which these flags are False)
    "flag_remove_cat",
    "flag_percent",
    "flag_value_no_unit",
    "flag_unit_nonstd",
    "flag_response_unit",
    "flag_unknown_subtype",
    "flag_all_hazards_unknown",
    "flag_pop_cntry",
    "flag_remove_unit",
    "flag_remove_hazard",
    "flag_failed_startYear_inference",
    "flag_inconsistent_year",
    "flag_country_location_missing",
]
labelled_df[filter_flags].sum()

flag_remove_cat                    50
flag_percent                        7
flag_value_no_unit                  0
flag_unit_nonstd                   17
flag_response_unit                  0
flag_unknown_subtype                7
flag_all_hazards_unknown           15
flag_pop_cntry                      0
flag_remove_unit                    0
flag_remove_hazard                 11
flag_failed_startYear_inference     0
flag_inconsistent_year              0
flag_country_location_missing       0
dtype: int64

In [59]:
print_impact(
    labelled_df[
        (labelled_df["flag_remove_hazard"] == True)
        | (labelled_df["flag_all_hazards_unknown"] == True)
        | (labelled_df["flag_unknown_subtype"] == True)
    ]
)

,appealCode,impactSubtype,impactValue,impactUnit,location,locationPolygon,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,annotation
437,MDRPK026,Human Deaths,306.0,people,[],[Pakistan],2024.0,6.0,NaN,NaN,NaN,NaN,[],[The monsoon season caused 306 fatalities and ...
438,MDRPK026,Injured People,584.0,people,[],[Pakistan],2024.0,6.0,NaN,NaN,NaN,NaN,[],[The monsoon season caused 306 fatalities and ...
439,MDRPK026,Transportation,40.0,roads,[],[Pakistan],2024.0,6.0,NaN,NaN,NaN,NaN,[],[The monsoon season caused 306 fatalities and ...
440,MDRPK026,Residential Buildings,20653.0,homes,[],[Pakistan],2024.0,6.0,NaN,NaN,NaN,NaN,[],[The monsoon season caused 306 fatalities and ...
441,MDRPK026,"Water, Sanitation, and Hygiene",NaN,null,[],[Pakistan],2024.0,6.0,1.0,NaN,NaN,NaN,[],"[As a result, urgent humanitarian needs emerge..."
442,MDRPK026,Human Health and Wellbeing,NaN,null,[],[Pakistan],2024.0,6.0,1.0,NaN,NaN,NaN,[],"[As a result, urgent humanitarian needs emerge..."
443,MDRPK026,Healthcare,NaN,null,[],[Pakistan],2024.0,6.0,1.0,NaN,NaN,NaN,[],"[Additionally, the stagnant water in low-lying..."
502,MDRSD034,Healthcare,80.0,healthcare structures,[],[Sudan],2024.0,NaN,NaN,2025.0,3.0,NaN,[Conflict],[The conflict disrupted the operations of 80% ...
506,MDRSD034,Undefined Infrastructure and Service Access,NaN,null,[],[Sudan],2024.0,NaN,NaN,2025.0,3.0,10.0,[Conflict],[The conflict caused significant damage to inf...
551,MDRSD034,Human Deaths,112.0,people,"[Kassala, Gedaref]","[Kassala, Sudan]",2024.0,7.0,22.0,2024.0,9.0,1.0,[Epidemic],"[From 22 July 2024 to September 1, 2024, MoH s..."


In [33]:
# filter unwanted data
from src.sanity_checks import flag_remove_cat
# remove_cats = ["Unknown","DREF Allocation", "Targeted People", "Assisted People", "Other Human Impacts", "Other Infrastructural Impacts", "Other Agricultural Impacts", "Other Service Access Impacts"]
# extracted_df["flag_remove_cat"] = extracted_df.apply(flag_remove_cat, remove_cats=remove_cats, axis=1)
# labelled_df["flag_remove_cat"] = labelled_df.apply(flag_remove_cat, remove_cats=remove_cats, axis=1)

filter_flags = (
    [  # flags to filter out (i.e. keep only rows for which these flags are False)
        "flag_remove_cat",
        "flag_percent",
        "flag_value_no_unit",
        "flag_unit_nonstd",
        "flag_response_unit",
        "flag_unknown_subtype",
        "flag_all_hazards_unknown",
        "flag_pop_cntry",
        "flag_remove_unit",
        "flag_remove_hazard",
        "flag_failed_startYear_inference",
        "flag_inconsistent_year",
        "flag_country_location_missing",
    ]
)
keep_vars = ["appealType", "reportSize", "reportDate"]
extracted_df = filter_by_flags(extracted_df, flag_filters=filter_flags)
labelled_df = filter_by_flags(labelled_df, flag_filters=filter_flags)

# combine data
combined_df = pd.concat([labelled_df, extracted_df])

In [34]:
extracted_df.shape

(738, 91)

In [35]:
labelled_df.shape

(673, 82)

In [36]:
labelled_df.appealCode.nunique()

39

In [37]:
extracted_df.appealCode.nunique()

39

## Match
1. Vectorize columns that need to be compared using cosine similarity
2. Compute cosine similarity for those columns for each possible extracted-labelled pair
3. Add absolute difference of impactValue between each possible extracted-labelled pair.
    Need to consider NaN from not NaN separately. Only try matching non-NaNs with non-NaNs 
    (and nans with nan?)
4. Compute Intersection-Over-Union of polygons for each possible pair
5. Match by maximizing similarity and -impactvalu_idff and -IoT. Allow for more than one match.  

In [38]:
# weights for matching
weight_set = dict()
weight_set["ws1"] = {#only impsubtype and unit
    'hazards' : 0,
    'iso3_code' : 0,
    'startYear' : 0,
    'startMonth' : 0,
    'startDay' : 0,
    'endYear' : 0,
    'endMonth' : 0,
    'endDay' : 0,
    'impactSubtype' : 1,
    'impactUnit' : 1,
    'geometry' : 0, #weight for geometry matching
    'impactValue' : 0
    }
weight_set["ws2"] = {#biased towards impsubtype and impunit
    'hazards' : 2,
    'iso3_code' : 2,
    'startYear' : 2,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 20, #bigger than sum of all other weights (24)
    'impactUnit' : 20,
    'geometry' : 5, #weight for geometry matching
    'impactValue' : 5
    }

weight_set["ws3"] = {#flat
    'hazards' : 1,
    'iso3_code' : 1,
    'startYear' : 1,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 1, #bigger than sum of all other weights (12)
    'impactUnit' : 1,
    'geometry' : 1, #weight for geometry matching
    'impactValue' : 1
    }
weight_set["ws4"] = {#only impsubtype
    'hazards' : 0,
    'iso3_code' : 0,
    'startYear' : 0,
    'startMonth' : 0,
    'startDay' : 0,
    'endYear' : 0,
    'endMonth' : 0,
    'endDay' : 0,
    'impactSubtype' : 1,
    'impactUnit' : 0,
    'geometry' : 0, #weight for geometry matching
    'impactValue' : 0
    }

weight_set["ws5"] = {#for quali data, subtype + geom > sum of other weights (12)
    'hazards' : 2,
    'iso3_code' : 2,
    'startYear' : 2,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 12,
    'impactUnit' : 0,
    'damageDegree' : 0,
    'geometry' : 4,
    'impactValue' : 0
    }
weight_set["ws6"] = {#for quanti data, 2 of impSub, impUn or impVal > sum of other weights (16)
    'hazards' : 2,
    'iso3_code' : 2,
    'startYear' : 2,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 12,
    'impactUnit' : 12,
    'damageDegree' : 1,
    'geometry' : 5,
    'impactValue' : 6
    }

weight_set["ws7"] = {  # quanti data; match (0.5) only if at least 2 of impSub, impUn or impVal match
        "hazards": 2,
        "iso3_code": 2,
        "startYear": 2,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 1,
        "endMonth": 1,
        "endDay": 1,
        "impactSubtype": 16,
        "impactUnit": 16,
        "damageDegree": 1,
        "geometry": 5,
        "impactValue": 16,
    }

# Optimized weight sets based on feature importance analysis
weight_set["ws_quanti_opt"] = {
    # Quantitative: prioritize value and unit matching
    "impactValue": 25,      # CRITICAL - core quantitative data
    "impactUnit": 20,       # CRITICAL - must match with value
    "impactSubtype": 12,    # HIGH - important but sometimes fuzzy
    "geometry": 6,          # MEDIUM - spatial confirmation
    "hazards": 3,           # LOW-MEDIUM - context
    "startYear": 3,         # LOW-MEDIUM - important but hard to extract
    "iso3_code": 2,         # LOW - context
    "damageDegree": 2,      # LOW - secondary
    "startMonth": 1,        # VERY LOW - difficult
    "startDay": 1,          # VERY LOW - difficult
    "endYear": 1,           # VERY LOW - secondary
    "endMonth": 1,          # VERY LOW - difficult
    "endDay": 1,            # VERY LOW - difficult
}

weight_set["ws_quali_opt"] = {
    # Qualitative: prioritize subtype and geometry (no value/unit available)
    "impactSubtype": 16,    # CRITICAL - no values available
    "geometry": 7,          # HIGH - spatial confirmation
    "hazards": 3,           # LOW-MEDIUM - context
    "startYear": 3,         # LOW-MEDIUM - important
    "iso3_code": 2,         # LOW - context
    "damageDegree": 0,      # NOT APPLICABLE - no damageDegree in qualitative
    "impactValue": 0,       # NOT APPLICABLE
    "impactUnit": 0,        # NOT APPLICABLE
    "startMonth": 1,        # VERY LOW - difficult
    "startDay": 1,          # VERY LOW - difficult
    "endYear": 1,           # VERY LOW - secondary
    "endMonth": 1,          # VERY LOW - difficult
    "endDay": 1,            # VERY LOW - difficult
}


In [39]:
#define target variables for (cosine) similarity calculation
UNIQUE_COUNTRIES_ISO = [country.alpha_3 for country in pycountry.countries]
UNIQUE_COUNTRY_NAMES = [country.name for country in pycountry.countries]

UNIQUE_DICT = {#mapping dictonary of unique values to generate vectors for cosine similarity
    'hazards' : list(hazard_main_types_emdat_desc.keys()),
    #'country' : UNIQUE_COUNTRY_NAMES,
    'iso3_code' : UNIQUE_COUNTRIES_ISO,
    'startYear' : np.arange(1980, 2026).tolist()+[-1],
    'startMonth' : np.arange(1, 13).tolist()+[-1],
    'startDay' : np.arange(1, 32).tolist()+[-1],
    'endYear' : np.arange(1980, 2026).tolist()+[-1],
    'endMonth' : np.arange(1, 13).tolist()+[-1],
    'endDay' : np.arange(1, 32).tolist()+[-1],
    'impactSubtype' : IMPACT_SUBTYPES_MERGED,
    'impactUnit' : combined_df.impactUnit.unique().tolist(),
}

SIMILARITY_VARS = list(UNIQUE_DICT.keys()) #all cols for which cosine similarity needs to be computed

In [40]:
#extracted_df = extracted_df[extracted_df["appealCode"]=="MDRS2001"]

In [41]:
## Parameters
geo_match = True
match_by = "labelled"
value_match = "pre"#"pre", "post" minimize diff of impactValue simultaneously as cat columns (pre) or after matching of cat columns (post)
impactValue_error_clip = None
nan_policy = "loose" # "loose", "strict" #allow NaNs from one side to be matched with all rows from the other side

#select variable for matching
matching_cols = list() + SIMILARITY_VARS #cols used for matching

#choose weights
ws_key_qt = "ws_quanti_opt"
ws_key_ql = "ws_quali_opt"
matching_cols_weights_qt = weight_set[ws_key_qt] # weights for matching quantitative
matching_cols_weights_ql = weight_set[ws_key_ql] # weights for matching qualitative

#updating matching columns and weights according to parameters
if geo_match:
    matching_cols.append("geometry")
if value_match == "pre":
    matching_cols.append("impactValue")
else:
    impactValue_error_clip = (0,1)

similarity_cols = [col for col in matching_cols if col in SIMILARITY_VARS]

#saving params
save_results = True
sim_name = f"{ws_key_qt}qt-{ws_key_ql}ql-geo-value{value_match}-{nan_policy}-nans"
filename_out = f"matched_data_by_{match_by}_{sim_name}_{res_savename_ext}"


In [42]:
##Matching
match_idx = []

for appeal, ext_group in extracted_df.groupby("appealCode"):
    print("Processing appeal", appeal)
    ext_group = ext_group.reset_index(drop=False, names=["orig_index"]) #need to reset index to get indices for numpy arrays
    lab_group = labelled_df[labelled_df["appealCode"] == appeal].reset_index(drop=False, names=["orig_index"])

    if (lab_group.shape[0] == 0) or (ext_group.shape[0] == 0):
        continue

    #vectorize
    ext_vect_df = pd.DataFrame(columns=similarity_cols)
    lab_vect_df = pd.DataFrame(columns=similarity_cols)

    for col in similarity_cols:
        ext_vect_df[col] = ext_group[col].apply(vectorize, unique_values=UNIQUE_DICT[col])
        lab_vect_df[col] = lab_group[col].apply(vectorize, unique_values=UNIQUE_DICT[col])

    #initialize
    reid_match_ext_group= np.array([])
    reid_match_lab_group = np.array([])
    accuracy_matrix_group = []

    #split between not nans and nans for impactValue
    not_nan_ext_df, not_nan_lab_df, nan_ext_df, nan_lab_df = split_nans(ext_group, lab_group, "impactValue", nan_policy=nan_policy)
    if len(not_nan_ext_df) and len(not_nan_lab_df):
        ext_vect_df_notna = ext_vect_df.loc[not_nan_ext_df.index]
        lab_vect_df_notna = lab_vect_df.loc[not_nan_lab_df.index]

        reid_match_ext, reid_match_lab, accuracy_matrix = match_rows(not_nan_ext_df, not_nan_lab_df, ext_vect_df_notna, lab_vect_df_notna, matching_cols, similarity_cols,  matching_cols_weights_qt, geo_match=geo_match, value_match=value_match, match_by=match_by)

        # Note: accuracy_matrix now contains columns in the order they appear in dist_mat:
        # similarity_cols + (geometry if geo_match) + (impactValue if value_match=='pre')
        # The aggregated similarity is computed internally by find_match_sim via compute_weighted_sim
        # Each labelled row is uniquely matched to one or more extracted rows (if similarities are equal)

        #store results for group
        accuracy_matrix_group.append(accuracy_matrix)
        reid_match_ext_group = np.append(reid_match_ext_group, reid_match_ext)
        reid_match_lab_group = np.append(reid_match_lab_group, reid_match_lab)

    if len(nan_ext_df) and len(nan_lab_df):

        ext_vect_df_na = ext_vect_df.loc[nan_ext_df.index]
        lab_vect_df_na = lab_vect_df.loc[nan_lab_df.index]

        #turn off value match for nans
        reid_match_ext, reid_match_lab, accuracy_matrix = match_rows(nan_ext_df, nan_lab_df, ext_vect_df_na, lab_vect_df_na, matching_cols, similarity_cols, matching_cols_weights_ql,  geo_match=geo_match, value_match=None, match_by="labelled")

        # Note: For NaN rows, value_match is None, so impactValue is not in dist_mat.
        # If value_match was 'pre' for non-NaN rows, we need to pad with NaN column for consistency
        if value_match == "pre":
            # insert NaN column before aggregated 'match' column to align with non-NaN ordering
            accuracy_matrix = np.insert(accuracy_matrix, accuracy_matrix.shape[1]-1, np.nan, axis=1)

        #store results for group
        accuracy_matrix_group.append(accuracy_matrix)
        reid_match_ext_group = np.append(reid_match_ext_group, reid_match_ext)
        reid_match_lab_group = np.append(reid_match_lab_group, reid_match_lab)

    #write as df
    # accuracy_matrix columns are: similarity_cols + optional columns (geometry, impactValue)
    # Build column names in the same order as they appear in dist_mat
    accuracy_cols = list(similarity_cols)
    if geo_match:
        accuracy_cols.append("geometry")

    if value_match == "pre":
        accuracy_cols.append("impactValue")
    # aggregated match similarity appended by `match_rows` as last column
    accuracy_cols.append("match")

    id_accuracy_array = np.append(np.stack((reid_match_ext_group, reid_match_lab_group),axis=1), np.concatenate(accuracy_matrix_group), axis=1)
    match_idx.append(pd.DataFrame(id_accuracy_array,
                                  columns = ["ext_match_id", "lab_match_id"] + accuracy_cols))

match_idx_df = pd.concat(match_idx)

# join extracted and labelled dataframes
matched_df = pd.concat([extracted_df.loc[match_idx_df["ext_match_id"].values].reset_index(drop=True),
                        labelled_df.loc[match_idx_df["lab_match_id"].values].add_suffix('_matched').reset_index(drop=True),
                        match_idx_df.reset_index(drop=True).add_suffix('_sim')], axis=1)

# overwrite / recompute impactValue sim and error
matched_df["impactValue_error"] = max_value_diff(matched_df["impactValue"].values, matched_df["impactValue_matched"].values)
# matched_df["impactValue_sim"] = calc_value_sim(matched_df["impactValue"].values, matched_df["impactValue_matched"].values)#[np.arange(len(matched_df)), np.arange(len(matched_df))]
if "geometry_sim" not in matched_df.columns:
    matched_df["geometry_sim"] = matched_df.apply(lambda x: IoU(x["geometry"], x["geometry_matched"]), axis=1)

if save_results:
    matched_df = delistify_cols(matched_df)
    matched_df.to_feather(DATA_OUT_PROC / (filename_out + ".feather"))

Processing appeal MDR55001
Processing appeal MDRAF014
Processing appeal MDRBD022
Processing appeal MDRBJ019
Processing appeal MDRCM036
Processing appeal MDRCM039
Processing appeal MDRCN006
Processing appeal MDRCO023
Processing appeal MDRDZ008
Processing appeal MDRDZ011
Processing appeal MDREC019
Processing appeal MDRGW003
Processing appeal MDRHU005
Processing appeal MDRID013
Processing appeal MDRIQ014
Processing appeal MDRIR009
Processing appeal MDRJO003
Processing appeal MDRKE058
Processing appeal MDRKZ010
Processing appeal MDRLB013
Processing appeal MDRMN017
Processing appeal MDRMY003
Processing appeal MDRMZ024
Processing appeal MDRNG041
Processing appeal MDRPG008
Processing appeal MDRPH036
Processing appeal MDRPK018
Processing appeal MDRPK026
Processing appeal MDRRS015
Processing appeal MDRRW022
Processing appeal MDRS2001
Processing appeal MDRSD034
Processing appeal MDRSV012
Processing appeal MDRTJ035
Processing appeal MDRUG050
Processing appeal MDRUY004
Processing appeal MDRVU012
P

In [43]:
filename_out

'matched_data_by_labelled_ws_quanti_optqt-ws_quali_optql-geo-valuepre-loose-nans_post_processed_0-39_labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v070726_v080726_geo'

In [44]:
matched_df

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,location,locationAnnotation,...,startDay_sim,endYear_sim,endMonth_sim,endDay_sim,impactSubtype_sim,impactUnit_sim,geometry_sim,impactValue_sim,match_sim,impactValue_error
0,Affected People,247408.0,NaN,NaN,exact,people,"['Number of people affected: 247,408']",0,[],"['Number of people affected: 247,408']",...,0.0,0.0,0.0,0.0,1.0,1.0,1.000000,1.0,0.927632,0.0
1,Education,3.0,NaN,NaN,exact,education structures,['Two out of three schools rehabilitations hav...,0,['Emae Island'],['Two out of three schools rehabilitations hav...,...,0.0,0.0,0.0,0.0,1.0,1.0,1.000000,1.0,0.894737,0.0
2,"Water, Sanitation, and Hygiene",NaN,NaN,NaN,None,null,['WASH recovery work begins in Vanuatu and Ope...,0,"['Paama', 'the Shepherds islands']","['In Vanuatu, WASH activities in Paama and the...",...,0.0,0.0,0.0,0.0,0.0,0.0,0.023234,NaN,0.112696,NaN
3,"Water, Sanitation, and Hygiene",NaN,NaN,NaN,None,null,['WASH recovery work begins in Vanuatu and Ope...,0,"['Paama', 'the Shepherds islands']","['In Vanuatu, WASH activities in Paama and the...",...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,NaN,0.138889,NaN
4,"Water, Sanitation, and Hygiene",NaN,NaN,NaN,None,null,"['In Vanuatu, WASH activities in Paama and the...",0,"['Paama', 'Shepherds islands']","['In Vanuatu, WASH activities in Paama and the...",...,0.0,0.0,0.0,0.0,0.0,0.0,0.023234,NaN,0.112696,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788,"Water, Sanitation, and Hygiene",NaN,NaN,NaN,None,null,['Water points are drying up making it dicult ...,0,"['Southern Province', 'Western Province', 'Nor...",['The whole Southern half of Zambia is experie...,...,0.0,0.0,0.0,0.0,1.0,1.0,1.000000,NaN,0.861111,NaN
789,"Water, Sanitation, and Hygiene",NaN,NaN,NaN,None,null,['Water points are drying up making it dicult ...,0,"['Southern Province', 'Western Province', 'Nor...",['The whole Southern half of Zambia is experie...,...,0.0,0.0,0.0,0.0,0.0,1.0,1.000000,NaN,0.416667,NaN
790,Agriculture and Access to Food,NaN,NaN,NaN,None,null,['Pasture is drying up making is dicult for li...,0,"['Western province', 'Southern Province', 'Cen...",['The provinces affected include North-Western...,...,0.0,0.0,0.0,0.0,1.0,1.0,0.743961,NaN,0.811326,NaN
791,Economy and Livelihood,NaN,NaN,NaN,None,null,['Price for food items and other basic commodi...,0,"['Southern Province', 'North-Western', 'Wester...",['The whole Southern half of Zambia is experie...,...,0.0,0.0,0.0,0.0,1.0,1.0,0.743961,NaN,0.811326,NaN
